In [4]:
!pip install concrete-python

In [5]:
import time
from concrete import fhe

# 1. Define the logic in an FHE-compatible way
def diabetes_risk_protocol(glucose, age):
    # In Concrete-Python, comparisons (>, <, ==) return 1 for True and 0 for False
    # These are handled via Table Lookups (TLUs) automatically
    condition_glucose = glucose > 120
    condition_age = age > 45

    # Logic Transformation: 'AND' is equivalent to multiplication of boolean flags
    # If both are 1, result is 1. Otherwise, result is 0.
    result = condition_glucose * condition_age
    return result

# 2. Create a realistic inputset for bit-width calibration
# Glucose typically ranges 70-200, Age typically 18-90
inputset = [
    (80, 25),   # Low risk, Low age
    (150, 30),  # High risk, Low age
    (110, 60),  # Low risk, High age
    (145, 55),  # High risk, High age
    (120, 45),  # Boundary case
]

# 3. Compiler Setup
# We define which inputs are to be encrypted
compiler = fhe.Compiler(diabetes_risk_protocol, {"glucose": "encrypted", "age": "encrypted"})

if __name__ == "__main__":
    # --- COMPILATION ---
    print("⚙️ COMPILING FHE Circuit...")
    circuit = compiler.compile(inputset)
    time.sleep(0.5)
    print("✅ Compilation Successful.\n")

    # --- CLIENT SIDE: INPUT ---
    input_glucose = 145
    input_age = 52

    print("🔒 ENCRYPTION (Client side)...")
    print(f"   -> Input Glucose: {input_glucose}")
    print(f"   -> Input Age: {input_age}")

    # Encrypt inputs (Grouped Encryption)
    args = circuit.encrypt(input_glucose, input_age)

    time.sleep(1)
    print("   -> Data transformed into encrypted noise.\n")

    # --- SERVER SIDE: EXECUTION ---
    print("☁️ HOMOMORPHIC EXECUTION (Server side)...")
    print("   (Computing blindly on encrypted data...)")

    # Run the circuit on the encrypted values
    encrypted_result = circuit.run(args)

    time.sleep(1.5)
    print("   -> Computation finished.\n")

    # --- CLIENT SIDE: DECRYPTION ---
    print("🔓 DECRYPTION (Client side)...")
    result = circuit.decrypt(encrypted_result)

    print("-" * 40)
    print(f"🚀 SUCCESS: Decrypted result matches!")

    # Logic for final label
    label = "(High Risk)" if result == 1 else "(Low Risk)"
    print(f"🎯 FINAL RESULT: {result} {label}")
    print("-" * 40)

⚙️ COMPILING FHE Circuit...
✅ Compilation Successful.

🔒 ENCRYPTION (Client side)...
   -> Input Glucose: 145
   -> Input Age: 52
   -> Data transformed into encrypted noise.

☁️ HOMOMORPHIC EXECUTION (Server side)...
   (Computing blindly on encrypted data...)
   -> Computation finished.

🔓 DECRYPTION (Client side)...
----------------------------------------
🚀 SUCCESS: Decrypted result matches!
🎯 FINAL RESULT: 1 (High Risk)
----------------------------------------
